# TFM BraTS-GLI — Entrenamiento en Colab (Swin-UNETR en L4)

Notebook alineado con el repositorio (arreglo de I/O con cache persistente, perfil `colab_l4.yaml` y `swin_unetr_l4.yaml` con gradient checkpointing).

**Antes de empezar:** en `Runtime > Change runtime type` selecciona GPU **L4** (o A100 si vas a usar `swin_unetr.yaml` + `colab_pro.yaml`).

El split `test` NO se toca aqui: solo se entrena y se evalua sobre `val`.

## 1. Comprobar GPU

In [ ]:
!nvidia-smi

## 2. Montar Google Drive
Estructura esperada: `/content/drive/MyDrive/TFM-datasets/{training_data1_v2, training_data_additional}`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Clonar el repositorio (privado -> requiere token)

El repo es privado, asi que GitHub no acepta usuario/contrasena por HTTPS. Usa un Personal Access Token (PAT):

1. GitHub > Settings > Developer settings > Personal access tokens > Fine-grained tokens. Acceso solo a este repo, permiso `Contents: Read-only`.
2. En Colab, panel izquierdo > Secrets (icono de la llave) > Add new secret. Name: `GITHUB_TOKEN`, Value: tu token. Activa el acceso para este notebook.
3. Ejecuta la celda de credenciales y luego la de clone.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
Path("/content/git_askpass.py").write_text(
    "#!/usr/bin/env python3\n"
    "import os, sys\n"
    "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['GITHUB_TOKEN'])\n",
    encoding="utf-8",
)
os.chmod("/content/git_askpass.py", 0o700)
os.environ["GIT_ASKPASS"] = "/content/git_askpass.py"
os.environ["GIT_TERMINAL_PROMPT"] = "0"
print("credenciales listas")

In [ ]:
%cd /content

!git clone https://github.com/jesusferron/tfm-brain-tumor-segmentation.git tfm-brain-tumor-segmentation || (cd tfm-brain-tumor-segmentation && git pull origin main)

%cd /content/tfm-brain-tumor-segmentation

!git log --oneline -3

!ls requirements/protocol.txt

## 4. Instalar dependencias

In [ ]:
%cd /content/tfm-brain-tumor-segmentation

!pip install -r requirements/protocol.txt

!python -c "import torch, monai, nibabel; print('torch', torch.__version__); print('cuda', torch.cuda.is_available())"

## 5. Arreglo de I/O: copiar el dataset al disco local del runtime
Drive es lento por acceso aleatorio. Copiamos los roots supervisados a `/content` (disco local). Una vez por sesion.
Combinado con el cache persistente (ya activado en las configs), tras la primera epoca la lectura deja de ser el cuello de botella.

In [ ]:
!mkdir -p /content/TFM-datasets

!rsync -ah --info=progress2 "/content/drive/MyDrive/TFM-datasets/training_data1_v2" /content/TFM-datasets/

!rsync -ah --info=progress2 "/content/drive/MyDrive/TFM-datasets/training_data_additional" /content/TFM-datasets/

### 5b. Apuntar `dataset_root` al disco local

In [ ]:
from pathlib import Path

import yaml



dataset_root = "/content/TFM-datasets"  # disco local del runtime, NO Drive

config_path = Path("configs/dataset/brats_gli_2024.yaml")

config = yaml.safe_load(config_path.read_text())

config["dataset_root"] = dataset_root

config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(config_path.read_text())

In [ ]:
# Verificar que se ven los dos roots supervisados

import yaml

from pathlib import Path

config = yaml.safe_load(Path("configs/dataset/brats_gli_2024.yaml").read_text())

root = Path(config["dataset_root"])

print("dataset_root:", root)

for name in config["training_roots"]:

    p = root / name

    print(p, "OK" if p.is_dir() else "MISSING")

## 6. QC rapido (3 casos)

In [ ]:
!python -m tfm_brats.cli qc \

  --dataset-config configs/dataset/brats_gli_2024.yaml \

  --output-csv outputs/qc/colab_qc_sample.csv \

  --output-json outputs/qc/colab_qc_sample_summary.json \

  --max-cases 3 --skip-intensity-stats --progress-every 1 --fail-on-problems

## 7. Smoke test de Swin-UNETR en L4 (2 pasos)
Valida carga, transforms, forward, loss, backward, validacion y checkpoint con la config L4 + gradient checkpointing.

In [ ]:
!python -m tfm_brats.cli train \

  --dataset-config configs/dataset/brats_gli_2024.yaml \

  --model-config configs/model/swin_unetr_l4.yaml \

  --training-config configs/training/colab_l4.yaml \

  --split-dir outputs/splits/brats_gli_2024_seed20260526 \

  --output-dir outputs/train/swin_unetr_l4_smoke \

  --max-steps 2 --max-train-cases 2 --max-val-cases 1 --device cuda

## 8. Entrenamiento real de Swin-UNETR en L4 (5000 pasos)
La primera epoca aun paga la lectura inicial mientras se construye el cache; a partir de la segunda, `data_wait_seconds` deberia caer.
Vigila en el log `gpu_memory_gb` (pico) y `step_seconds`. Si hay OOM: baja `patch_size` a `[96,96,96]` en `colab_l4.yaml`, o pasa a A100 con `swin_unetr.yaml` + `colab_pro.yaml`.

In [ ]:
!python -m tfm_brats.cli train \

  --dataset-config configs/dataset/brats_gli_2024.yaml \

  --model-config configs/model/swin_unetr_l4.yaml \

  --training-config configs/training/colab_l4.yaml \

  --split-dir outputs/splits/brats_gli_2024_seed20260526 \

  --output-dir outputs/train/swin_unetr_l4 \

  --max-steps 5000 --device cuda

In [ ]:
!tail -n 20 outputs/train/swin_unetr_l4/train_log.csv

## 9. Predecir sobre `val` con `best.pt`

In [ ]:
!python -m tfm_brats.cli predict \

  --dataset-config configs/dataset/brats_gli_2024.yaml \

  --model-config configs/model/swin_unetr_l4.yaml \

  --training-config configs/training/colab_l4.yaml \

  --split-csv outputs/splits/brats_gli_2024_seed20260526/val.csv \

  --checkpoint outputs/train/swin_unetr_l4/checkpoints/best.pt \

  --output-dir outputs/predictions/swin_unetr_l4_val --device cuda

## 10. Evaluar `val` (Dice y HD95 por ET/TC/WT)

In [ ]:
!python -m tfm_brats.cli evaluate \

  --dataset-config configs/dataset/brats_gli_2024.yaml \

  --split-csv outputs/splits/brats_gli_2024_seed20260526/val.csv \

  --predictions-dir outputs/predictions/swin_unetr_l4_val \

  --output-csv outputs/evaluation/swin_unetr_l4_val_metrics.csv \

  --output-json outputs/evaluation/swin_unetr_l4_val_metrics_summary.json

!cat outputs/evaluation/swin_unetr_l4_val_metrics_summary.json

## 11. Empaquetar artefactos para descargar

In [ ]:
!zip -r swin_unetr_l4_results.zip \

  outputs/train/swin_unetr_l4/train_summary.json \

  outputs/train/swin_unetr_l4/train_log.csv \

  outputs/train/swin_unetr_l4/checkpoints/best.pt \

  outputs/evaluation/swin_unetr_l4_val_metrics.csv \

  outputs/evaluation/swin_unetr_l4_val_metrics_summary.json

from google.colab import files

files.download('swin_unetr_l4_results.zip')